# Charlotte Project | Week 10 — Class 3
## Pandas Selection, Filtering and Multi-Level Aggregation

**Mode:** Interactive Jupyter Notebook | **Duration:** 180 minutes | **Assessment:** Independent `.py` scripts and GitHub Pull Request

**Learning objectives:** combine multiple conditions with `.loc`, distinguish `.loc` and `.iloc`, avoid chained-assignment errors, summarize wells and fields using `groupby` and named `agg`, reshape with `pivot_table`, and interpret rankings and missing values.

**Important:** This notebook uses **demonstration readings for four teaching wells**. The assessed lab uses a separate 24-record CSV for four different wells. Run the teaching notebook, then complete your lab independently in a Python script.

## A. Accurate row and column selection (35 minutes)

**Concept and why it matters.** A production DataFrame has one row per operational observation and one column per attribute (well, field, date, status, oil, water, pressure). Selecting the correct subset is the foundation of reliable analysis: choosing the wrong rows can make inactive assets appear productive or can hide missing values.

**Key language.** `.loc[row_labels, column_labels]` uses *labels* and supports Boolean masks; `.iloc[row_positions, column_positions]` uses *integer positions*. A condition such as `df["oil_bbl"] > 100` creates a Boolean Series. Use parentheses around each comparison and `&` (and), `|` (or), `~` (not) to combine Series; Python `and`/`or` cannot combine whole Pandas Series.

**Common mistakes:** confusing position with label after filtering, omitting parentheses, and editing a slice without `.copy()`. Filtering should preserve missing readings for review, not quietly turn them into zeros.

In [ ]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)
demo = pd.DataFrame({
 "field": ["East","East","East","East","West","West","West","West","West"],
 "well_id": ["DEMO-E1","DEMO-E1","DEMO-E2","DEMO-E2","DEMO-W1","DEMO-W1","DEMO-W1","DEMO-W2","DEMO-W2"],
 "date": ["2026-09-01","2026-09-02","2026-09-01","2026-09-02","2026-09-01","2026-09-02","2026-09-03","2026-09-01","2026-09-02"],
 "status": ["ACTIVE","ACTIVE","ACTIVE","SHUT_IN","ACTIVE","ACTIVE","ACTIVE","ACTIVE","ACTIVE"],
 "oil_bbl": [110,115,80,0,160,170,np.nan,125,130],
 "water_bbl": [30,32,60,0,25,27,28,55,59],
 "pressure_psi": [2400,2420,2100,2095,2750,2760,2775,2500,2510]})
demo

### A1. Selecting by labels and positions

Read `.loc[rows, columns]` literally: first choose the *records*, then choose the *fields to display*. For repeatable business logic, choose columns by names rather than by position. `.iloc` is useful for quickly inspecting the first rows, but positions move when rows are sorted or inserted. Try predicting the result before running each cell.

In [ ]:
print("Label-based:")
print(demo.loc[0:2,["well_id","oil_bbl"]])
print("Position-based:")
print(demo.iloc[:3,[1,4]])

### A2. Combining conditions correctly

A production-surveillance question often has several conditions: “show active wells in the East field with more than 100 barrels of oil.” Each condition returns a True/False value per row. The `&` operator combines these values element-wise. Always wrap each comparison in parentheses. Use `.copy()` when you plan to calculate a new column on the filtered subset.

In [ ]:
mask=(demo["status"]=="ACTIVE") & (demo["field"]=="East") & (demo["oil_bbl"]>100)
selected=demo.loc[mask,["well_id","date","oil_bbl"]].copy()
print(selected)
print("Number selected:",len(selected))

### A3. Mini practice

Before running the next cell: predict which observations qualify when pressure is at least 2,500 psi, status is ACTIVE, and oil is not missing. Then change `2500` to `2700`. Why does the result change?

In [ ]:
high_pressure=demo.loc[(demo["pressure_psi"]>=2500)&(demo["status"]=="ACTIVE")&demo["oil_bbl"].notna(),["well_id","pressure_psi","oil_bbl"]]
print(high_pressure)

## B. Sorting, calculated columns, and data quality (30 minutes)

**Concept.** DataFrame calculations operate column-wise; a calculated column can express a domain metric for every row at once. Here `liquid_bbl = oil_bbl + water_bbl` and `water_cut_pct = 100 × water / liquid`. Water cut describes the fraction of water in the produced liquid, not the fraction of a reservoir filled with water.

**Why it matters.** Operational reports need consistent derived measurements and transparent treatment of incomplete records. A missing oil value yields missing liquid and water cut; it should not be silently interpreted as zero production. Also guard against division by zero. `sort_values()` orders records for review and ranking, but sorting alone is not aggregation.

**Common mistakes:** calculating water cut with `oil_bbl` as the denominator, computing a percentage on empty liquid volumes, and using chained assignment on filtered rows.

In [ ]:
quality=demo.copy()
quality["liquid_bbl"]=quality["oil_bbl"]+quality["water_bbl"]
quality["water_cut_pct"]=(100*quality["water_bbl"] / quality["liquid_bbl"].where(quality["liquid_bbl"]!=0)).round(2)
print(quality[["well_id","oil_bbl","water_bbl","liquid_bbl","water_cut_pct"]])
print("Missing oil rows:",quality["oil_bbl"].isna().sum())

### B1. Sort and inspect

Use `sort_values(["field", "oil_bbl"], ascending=[True, False])` to group fields alphabetically while placing larger oil readings first within each field. Missing values normally appear last; report them rather than hiding them.

In [ ]:
print(quality.sort_values(["field","oil_bbl"],ascending=[True,False])[["field","well_id","oil_bbl"]])

## C. GroupBy and named aggregation (45 minutes)

**Concept.** Grouping partitions rows that share the same keys, then aggregates each group. In energy surveillance, grouping by `well_id` yields per-well metrics while grouping by `field` yields a field-level summary. Grouping by both `field` and `well_id` produces a *multi-key* summary. The operation follows **split → apply → combine**.

**Important distinction:** `sum` answers “how much in total?”, `mean` answers “what was the average reading?”, `count` counts non-null measurements, `size` counts all records, and `nunique` counts distinct wells. A missing oil reading affects `count` but not `size`; without an explicit validation rule, Pandas sum may skip missing values.

**Syntax:** `df.groupby(["field","well_id"], as_index=False).agg(oil_total=("oil_bbl","sum"), mean_pressure=("pressure_psi","mean"))`. Named aggregation produces predictable column names. The key ordering matters for subsequent sort and joins.

**Common mistakes:** treating average water cut as the ratio of summed volumes (these are different definitions), confusing row counts with well counts, and accidentally including shut-in readings in producing-well totals.

In [ ]:
active=quality.loc[(quality["status"]=="ACTIVE") & quality[["oil_bbl","water_bbl"]].notna().all(axis=1)].copy()
print("Active + complete readings:",len(active))
well_report=active.groupby(["field","well_id"],as_index=False).agg(
 records=("well_id","size"),oil_total_bbl=("oil_bbl","sum"),
 water_total_bbl=("water_bbl","sum"),avg_pressure_psi=("pressure_psi","mean"),
 mean_water_cut_pct=("water_cut_pct","mean"))
well_report

### C1. From wells to fields

Different management questions require different grouping levels. The field manager may want total oil by field and the number of producing wells. Use `nunique` for distinct wells and not `size`, because one well may contribute many daily observations.

In [7]:
field_report=active.groupby("field",as_index=False).agg(
 distinct_wells=("well_id","nunique"),
 active_records=("well_id","size"),oil_total_bbl=("oil_bbl","sum"),
 liquid_total_bbl=("liquid_bbl","sum"))
print(field_report)

  field  distinct_wells  active_records  oil_total_bbl  liquid_total_bbl
0  East               2               3          305.0             427.0
1  West               2               4          585.0             751.0


### C2. Grouping with a hierarchical index

By default, `groupby(["field", "well_id"])` creates a MultiIndex. That is useful for hierarchical reporting, but a flat table is usually easier to export as CSV. Use `.reset_index()` or pass `as_index=False` during aggregation. Compare the layouts below and explain which would be easier to share with another system.

In [ ]:
hierarchy=active.groupby(["field","well_id"])["oil_bbl"].sum()
print("MultiIndex result:",hierarchy)
print("Flat result:",hierarchy.reset_index(name="oil_total_bbl"))

## D. Transform, ranking and pivot tables (40 minutes)

**Concept.** Aggregation reduces the number of rows: many daily observations become one result per well. `groupby().transform()` instead calculates a statistic for each group and broadcasts it back to every original row, preserving record-level detail. For example, every observation can show its well’s average pressure beside its own pressure.

**Ranking** orders wells by production and requires an explicit tie policy. `sort_values()` is enough for a top-well list; `rank(method="dense")` is useful when equal readings should share the same rank.

**Pivot tables** arrange grouped measures in a matrix, e.g., fields as rows and well IDs as columns. They are useful for compact operational comparisons. A pivot table does not automatically correct missing records or inconsistent well labels.

**Common mistakes:** assuming `transform()` shrinks rows, overlooking the chosen aggregation function in a pivot, and interpreting demo rankings as formal production performance guarantees.

In [ ]:
active["well_avg_pressure_psi"]=active.groupby("well_id")["pressure_psi"].transform("mean")
active["pressure_minus_well_avg"]=active["pressure_psi"]-active["well_avg_pressure_psi"]
print(active[["well_id","pressure_psi","well_avg_pressure_psi","pressure_minus_well_avg"]].head())

In [ ]:
ranking=well_report.sort_values(["oil_total_bbl","well_id"],ascending=[False,True]).copy()
ranking["dense_rank"]=ranking["oil_total_bbl"].rank(method="dense",ascending=False).astype(int)
print(ranking[["well_id","oil_total_bbl","dense_rank"]])

In [ ]:
matrix=active.pivot_table(index="field",columns="well_id",values="oil_bbl",aggfunc="sum")
print(matrix)
print("Notice: missing combinations remain NaN rather than becoming invented zeros.")

## E. Interpretation and independent assignment (30 minutes)

**From calculation to decision.** A defensible field surveillance report must name the filter, the grouping level, the units, and how missing or shut-in readings were treated. Explain how many observations were excluded and why. Your summary should be reproducible from raw source records.

**Do not submit this notebook as your lab.** Your lab dataset contains four different well IDs and its own missing readings. Complete the `.py` starter file, generate three CSV outputs, and submit a Pull Request from `feature/class-03-aggregation` in your **individual** GitHub repository. Do not commit the instructor solution, environment directory, or secrets.

**Knowledge checks:** (1) What is the difference between `.loc` and `.iloc`? (2) Why do `count` and `size` differ with nulls? (3) When would you choose `transform` over `agg`? (4) Why can a field total hide poor data quality? (5) How would you tie-break equally productive wells?

In [ ]:
print("Review checks")
print("Demo observations:",len(demo))
print("Accepted teaching observations:",len(active))
print("Summarized wells:",len(well_report))
print("Summarized fields:",len(field_report))
assert len(demo)==9 and len(active)==7 and len(well_report)==4 and len(field_report)==2
print("Teaching notebook checks passed")